In [1]:
print('test')

test


In [8]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/6r/qn2nw9l17zs4b7s1wnb0v8q40000gn/T/tmpcfd_yp5l
Processing /var/folders/6r/qn2nw9l17zs4b7s1wnb0v8q40000gn/T/tmpcfd_yp5l/pip-25.0.1-py3-none-any.whl


In [9]:
!{sys.executable} -m pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


In [20]:
import sys
!{sys.executable} -m pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 26.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


In [32]:
import io
import requests
import pandas as pd
import numpy as np
import lxml
import yfinance as yf

In [11]:
URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}

In [14]:
def fetch_sp500() -> pd.DataFrame:
    """Return a DataFrame of ticker, name, and year added."""
    response = requests.get(URL, headers=HEADERS, timeout=30)
    response.raise_for_status()
 
    # The first table on the page is the constituents list.
    tables = pd.read_html(io.StringIO(response.text))
    df = tables[0]
 
    # Column names as they appear on the page (as of 2025/2026).
    # Symbol | Security | GICS Sector | ... | Date added | CIK | Founded
    df = df.rename(columns={
        "Symbol": "ticker",
        "Security": "name",
        "Date added": "date_added",
    })
 
    df = df[["ticker", "name", "date_added"]].copy()
 
    # "Date added" is usually YYYY-MM-DD but a few rows carry footnotes or
    # blanks. Parse leniently and pull out the year.
    parsed = pd.to_datetime(df["date_added"], errors="coerce")
    df["year_added"] = parsed.dt.year
 
    return df


In [17]:
df = fetch_sp500()
df.head()


,ticker,name,date_added,year_added
0,MMM,3M,1957-03-04,1957
1,AOS,A. O. Smith,2017-07-26,2017
2,ABT,Abbott Laboratories,1957-03-04,1957
3,ABBV,AbbVie,2012-12-31,2012
4,ACN,Accenture,2011-07-06,2011


In [16]:
df["year_added"].isna().sum()   # how many dates didn't parse
 

np.int64(0)

In [18]:
def additions_per_year(df: pd.DataFrame, start_year: int = 2020) -> pd.Series:
    """Count additions per year, starting from `start_year`."""
    valid = df.dropna(subset=["year_added"]).copy()
    valid["year_added"] = valid["year_added"].astype(int)
    recent = valid[valid["year_added"] >= start_year]
    return recent["year_added"].value_counts().sort_index()

In [19]:
counts = additions_per_year(df, start_year=2020)
counts

year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

2025 had the highest number of additions to the S&P 500

In [22]:
# label -> Yahoo ticker
INDEXES = {
    "US - S&P 500":            "^GSPC",
    "China - Shanghai Comp":   "000001.SS",
    "Hong Kong - Hang Seng":   "^HSI",
    "Australia - ASX 200":     "^AXJO",
    "India - Nifty 50":        "^NSEI",
    "Canada - TSX Composite":  "^GSPTSE",
    "Germany - DAX":           "^GDAXI",
    "UK - FTSE 100":           "^FTSE",
    "Japan - Nikkei 225":      "^N225",
    "Mexico - IPC Mexico":     "^MXX",
    "Brazil - Ibovespa":       "^BVSP",
}
 
START = "2026-01-01"
LAST_DAY = "2026-08-21"
# yfinance's `end` is EXCLUSIVE, so to include the 21 Aug close we request
# through the 22nd, then slice to the last trading day on/before the 21st.
END = "2026-08-22"


In [24]:
df = yf.download("^GSPC", start="2026-01-01", end="2026-08-22",
                 progress=False, auto_adjust=True)
print(df.shape)
print("first:", df.index.min().date(), " last:", df.index.max().date())
df["Close"].tail(3)

(160, 5)
first: 2026-01-02  last: 2026-08-21


Ticker,^GSPC
Date,
2026-08-19,7707.979980
2026-08-20,7641.160156
2026-08-21,7674.370117


In [25]:
def ytd_return(ticker):
    df = yf.download(ticker, start="2026-01-01", end="2026-08-22",
                     progress=False, auto_adjust=True)
    if df.empty:
        return float("nan")
    close = df["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    close = close.loc[:"2026-08-21"].dropna()
    if len(close) < 2:
        return float("nan")
    return (close.iloc[-1] / close.iloc[0] - 1) * 100

In [26]:
print("US:", ytd_return("^GSPC"))
print("Japan:", ytd_return("^N225"))

US: 11.896237452163927
Japan: 27.36405978447678


In [27]:
results = {label: ytd_return(tkr) for label, tkr in INDEXES.items()}
s = pd.Series(results, name="YTD %").sort_values(ascending=False)
s.round(2)

Japan - Nikkei 225        27.36
Canada - TSX Composite    14.86
US - S&P 500              11.90
UK - FTSE 100              8.70
Brazil - Ibovespa          6.54
Germany - DAX              6.51
Australia - ASX 200        3.79
Mexico - IPC Mexico        2.48
Hong Kong - Hang Seng     -1.25
China - Shanghai Comp     -2.94
India - Nifty 50          -7.25
Name: YTD %, dtype: float64

In [28]:
us = s["US - S&P 500"]
better = s[s > us].drop("US - S&P 500", errors="ignore")
print(f"US (S&P 500) YTD: {us:.2f}%")
print(f"Indexes beating the US: {len(better)} out of {len(INDEXES) - 1}")
print(list(better.index))

US (S&P 500) YTD: 11.90%
Indexes beating the US: 2 out of 10
['Japan - Nikkei 225', 'Canada - TSX Composite']


US (S&P 500) YTD: 11.90% |
Indexes beating the US: 2 out of 10 - Japan & Canada

In [29]:
df = yf.download("^GSPC", start="1950-01-01", progress=False, auto_adjust=True)
print(df.shape)
print("range:", df.index.min().date(), "→", df.index.max().date())
df.tail(3)

(19293, 5)
range: 1950-01-03 → 2026-09-09


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2026-09-04,7718.600098,7750.189941,7706.120117,7750.189941,4103570000
2026-09-08,7673.520020,7717.810059,7666.990234,7717.810059,4966930000
2026-09-09,7645.549805,7660.680176,7624.160156,7660.680176,1420994000


In [30]:
close = df["Close"]
if isinstance(close, pd.DataFrame):   # single-ticker download can be 2-D
    close = close.iloc[:, 0]
close = close.dropna()
close.tail(3)

Date
2026-09-04    7718.600098
2026-09-08    7673.520020
2026-09-09    7645.549805
Name: ^GSPC, dtype: float64

In [33]:
running_max = close.cummax()
is_ath = close == running_max          # a day at (or above) every prior close
ath_positions = np.flatnonzero(is_ath.values)
print("all-time-high days:", len(ath_positions))
close[is_ath].tail(3)

all-time-high days: 1537


Date
2026-08-04    7736.520020
2026-08-07    7757.640137
2026-08-13    7798.990234
Name: ^GSPC, dtype: float64

In [34]:
prices = close.values
dates = close.index
records = []

for a, b in zip(ath_positions[:-1], ath_positions[1:]):
    segment = prices[a:b + 1]              # peak → recovery, inclusive
    peak = prices[a]
    trough = segment.min()
    drawdown = (peak - trough) / peak * 100
    duration = (dates[b] - dates[a]).days  # calendar days, peak to recovery
    records.append({
        "peak_date": dates[a],
        "recovery_date": dates[b],
        "peak": peak,
        "trough": trough,
        "drawdown_pct": drawdown,
        "duration_days": duration,
    })

corr = pd.DataFrame(records)
print(len(corr), "segments between consecutive highs")
corr.head()

1536 segments between consecutive highs


,peak_date,recovery_date,peak,trough,drawdown_pct,duration_days
0,1950-01-03,1950-01-04,16.66,16.660000,0.000000,1
1,1950-01-04,1950-01-05,16.85,16.850000,0.000000,1
2,1950-01-05,1950-01-06,16.93,16.930000,0.000000,1
3,1950-01-06,1950-01-09,16.98,16.980000,0.000000,3
4,1950-01-09,1950-01-11,17.08,17.030001,0.292736,2


In [35]:
corrections = corr[corr["drawdown_pct"] >= 5].copy()
print(len(corrections), "corrections of at least 5%")
corrections.sort_values("drawdown_pct", ascending=False).head(10)

74 corrections of at least 5%


,peak_date,recovery_date,peak,trough,drawdown_pct,duration_days
1067,2007-10-09,2013-03-28,1565.150024,676.530029,56.775388,1997
1058,2000-03-24,2007-05-30,1527.459961,776.760010,49.146948,2623
554,1973-01-11,1980-07-17,120.239998,62.279999,48.203593,2744
519,1968-11-29,1972-03-06,108.370003,69.290001,36.061641,1193
1322,2020-02-19,2020-08-18,3386.149902,2237.399902,33.924960,181
731,1987-08-25,1989-07-26,336.769989,223.919998,33.509515,701
348,1961-12-12,1963-09-03,72.639999,52.320000,27.973568,630
578,1980-11-28,1982-11-03,140.520004,102.419998,27.113582,705
1413,2022-01-03,2024-01-19,4796.560059,3577.030029,25.425097,746
471,1966-02-09,1967-05-04,94.059998,73.199997,22.177335,449


In [36]:
q = [0.25, 0.50, 0.75]
summary = pd.DataFrame({
    "drawdown_pct": corrections["drawdown_pct"].quantile(q),
    "duration_days": corrections["duration_days"].quantile(q),
})
summary.index = ["25th", "50th (median)", "75th"]
print(summary.round(2))
print(f"\nMedian drawdown: {corrections['drawdown_pct'].median():.2f}%")

               drawdown_pct  duration_days
25th                   6.23           55.5
50th (median)          7.99           92.5
75th                  14.02          211.5

Median drawdown: 7.99%


               drawdown_pct  duration_days
25th                   6.23           55.5
50th (median)          7.99           92.5
75th                  14.02          211.5

Median drawdown: 7.99%

In [37]:
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)

# default limit is 12; ask for more so we get the full history back to 2020
earnings = ticker_obj.get_earnings_dates(limit=60)
print(earnings.shape)
earnings.head()

(100, 3)


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-29 16:00:00-04:00,1.95,NaN,NaN
2026-07-30 16:00:00-04:00,1.83,5.75,215.02
2026-04-29 16:00:00-04:00,1.65,2.78,68.18
2026-02-05 16:00:00-05:00,1.95,1.95,0.22
2025-10-30 16:00:00-04:00,1.56,1.95,25.20


In [38]:
print(earnings.columns.tolist())

# the surprise column name varies slightly across versions ("Surprise(%)")
surprise_col = [c for c in earnings.columns if "Surprise" in c][0]
print("surprise column:", surprise_col)

# the upcoming earnings date has no Reported EPS / Surprise yet — drop it
earnings = earnings.dropna(subset=["Reported EPS"])
print("reported events:", len(earnings))
earnings[[surprise_col, "Reported EPS", "EPS Estimate"]].head()

['EPS Estimate', 'Reported EPS', 'Surprise(%)']
surprise column: Surprise(%)
reported events: 92


,Surprise(%),Reported EPS,EPS Estimate
Earnings Date,,,
2026-07-30 16:00:00-04:00,215.02,5.75,1.83
2026-04-29 16:00:00-04:00,68.18,2.78,1.65
2026-02-05 16:00:00-05:00,0.22,1.95,1.95
2025-10-30 16:00:00-04:00,25.20,1.95,1.56
2025-07-31 16:00:00-04:00,26.13,1.68,1.33


In [39]:
px = yf.download(ticker, start="2020-01-01", progress=False, auto_adjust=True)
close = px["Close"]
if isinstance(close, pd.DataFrame):     # single-ticker download can be 2-D
    close = close.iloc[:, 0]
close = close.dropna()
print(close.shape, close.index.min().date(), "→", close.index.max().date())

(1680,) 2020-01-02 → 2026-09-09


In [40]:
# For each Day2, return = Close_Day3 / Close_Day1 - 1
#   Day1 = previous trading day, Day3 = next trading day
ret2 = close.shift(-1) / close.shift(1) - 1
ret2.name = "ret2"
ret2.dropna().head()

Date
2020-01-03    0.002566
2020-01-06    0.017008
2020-01-07   -0.005733
2020-01-08   -0.003047
2020-01-09   -0.004657
Name: ret2, dtype: float64

In [41]:
# earnings index is tz-aware with a time-of-day; normalize to a naive date
edates = earnings.copy()
edates.index = pd.to_datetime(edates.index)
try:
    edates.index = edates.index.tz_localize(None)   # drop tz if present
except TypeError:
    pass
edates.index = edates.index.normalize()

r = ret2.copy()
r.index = r.index.normalize()

edates["ret2"] = r.reindex(edates.index).values
print("matched:", edates["ret2"].notna().sum(), "of", len(edates))
edates[[surprise_col, "ret2"]].sort_index()

matched: 27 of 92


,Surprise(%),ret2
Earnings Date,,
2002-04-23,12.94,NaN
2002-07-23,23.30,NaN
2003-01-23,31.03,NaN
2003-04-24,140.38,NaN
2003-07-22,64.47,NaN
...,...,...
2025-07-31,26.13,-0.067075
2025-10-30,25.20,0.060443
2026-02-05,0.22,-0.097300


In [44]:
# The task expects 25 entries starting 2020-10-29. Our pull went back to 2002,
# so two pre-Oct-2020 events slipped in. Restrict to the intended window.
edates = edates[edates.index >= pd.Timestamp("2020-10-29")]
print("events in scope:", len(edates))          # expect ~24 reported
print("range:", edates.index.min().date(), "→", edates.index.max().date())

events in scope: 24
range: 2020-10-29 → 2026-07-30


In [47]:
pos = edates[edates[surprise_col] > 0].dropna(subset=["ret2"])
print(f"positive-surprise events: {len(pos)}")
print(f"median 2-day return after positive surprises: {pos['ret2'].median() * 100:.2f}%")
pos[[surprise_col, "ret2"]].sort_index()

positive-surprise events: 20
median 2-day return after positive surprises: 0.35%


,Surprise(%),ret2
Earnings Date,,
2020-10-29,66.17,-0.040038
2021-02-02,97.39,-0.009079
2021-04-29,64.11,0.002579
2021-07-29,22.57,-0.083389
2022-02-03,641.94,0.046656
2022-10-27,33.00,-0.105914
2023-04-27,46.82,0.004477
2023-08-03,91.19,0.088605
2023-10-26,62.22,0.052311


In [46]:
both = edates.dropna(subset=["ret2", surprise_col])
corr_matrix = both[[surprise_col, "ret2"]].corr()
print(corr_matrix.round(3))
print(f"\ncorrelation (all events): {corr_matrix.loc[surprise_col, 'ret2']:.3f}")

             Surprise(%)   ret2
Surprise(%)        1.000  0.222
ret2               0.222  1.000

correlation (all events): 0.222


In [48]:
# Broad-market regime: S&P 500 above/below its 200-day moving average.
spx = yf.download("^GSPC", start="2019-06-01", progress=False, auto_adjust=True)
spx_close = spx["Close"]
if isinstance(spx_close, pd.DataFrame):
    spx_close = spx_close.iloc[:, 0]

sma200 = spx_close.rolling(200).mean()
regime = pd.Series(np.where(spx_close > sma200, "bull", "bear"),
                   index=spx_close.index)
regime.index = regime.index.normalize()
regime.tail(3)

Date
2026-09-04    bull
2026-09-08    bull
2026-09-09    bull
dtype: str

In [49]:
# most recent regime reading on or before each earnings date
reg_aligned = regime.reindex(edates.index, method="ffill")
edates = edates.assign(regime=reg_aligned.values)
edates[[surprise_col, "ret2", "regime"]]

,Surprise(%),ret2,regime
Earnings Date,,,
2026-07-30,215.02,0.198235,bull
2026-04-29,68.18,0.020639,bull
2026-02-05,0.22,-0.097300,bull
2025-10-30,25.20,0.060443,bull
2025-07-31,26.13,-0.067075,bull
2025-05-01,17.08,0.030149,bear
2025-02-06,25.36,-0.029724,bull
2024-10-31,25.17,0.026981,bull
2024-08-01,23.76,-0.102043,bull


In [50]:
summary = edates.dropna(subset=["ret2"]).groupby("regime").agg(
    n=("ret2", "size"),
    median_ret2=("ret2", "median"),
    mean_ret2=("ret2", "mean"),
    corr_surprise=(surprise_col, lambda x: x.corr(edates.loc[x.index, "ret2"])),
)
print(summary.round(4))

         n  median_ret2  mean_ret2  corr_surprise
regime                                           
bear     5       0.0301    -0.0017        -0.2134
bull    19      -0.0059     0.0049         0.3562
